In [1]:
import pandas as pd
from urllib.parse import quote
from pathlib import Path

RAW_DIR = Path('../Data/Raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Setup OK. Raw folder:", RAW_DIR.resolve())


Setup OK. Raw folder: D:\Library Analytics\Data\Raw


In [2]:
base = "https://data.seattle.gov/resource/tmmm-ytt6.csv"

where_clause = (
    "checkoutyear >= 2020 AND checkoutyear <= 2023 "
    "AND materialtype in ('BOOK', 'EBOOK', 'AUDIOBOOK') "
    "AND checkouts >= 5"
)

url = f"{base}?$where={quote(where_clause)}&$limit=2000000"

print("Pulling from API... 1-3 min is normal.")
df = pd.read_csv(url)
print(f"Done. Pulled {len(df):,} rows.")


Pulling from API... 1-3 min is normal.


C:\Users\vardh\AppData\Local\Temp\ipykernel_16232\4133834653.py:12: DtypeWarning: Columns (0: isbn) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url)


Done. Pulled 1,342,223 rows.


In [3]:
raw_path = RAW_DIR / 'seattle_checkouts_2020_2023.csv'
df.to_csv(raw_path, index=False)
print(f"Saved to: {raw_path.resolve()}")
print(f"File size: {raw_path.stat().st_size / 1e6:.1f} MB")

Saved to: D:\Library Analytics\Data\Raw\seattle_checkouts_2020_2023.csv
File size: 310.4 MB


In [4]:
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nDtypes:")
print(df.dtypes)
print("\nMemory (MB):", round(df.memory_usage(deep=True).sum() / 1e6, 1))
print("\nYear range:", df['checkoutyear'].min(), "-", df['checkoutyear'].max())
print("\nMaterialType counts:")
print(df['materialtype'].value_counts())
print("\nHead:")
df.head()


Shape: (1342223, 12)

Columns: ['usageclass', 'checkouttype', 'materialtype', 'checkoutyear', 'checkoutmonth', 'checkouts', 'title', 'isbn', 'creator', 'subjects', 'publisher', 'publicationyear']

Dtypes:
usageclass            str
checkouttype          str
materialtype          str
checkoutyear        int64
checkoutmonth       int64
checkouts           int64
title                 str
isbn               object
creator               str
subjects              str
publisher             str
publicationyear       str
dtype: object

Memory (MB): 891.9

Year range: 2020 - 2023

MaterialType counts:
materialtype
EBOOK        515600
BOOK         446506
AUDIOBOOK    380117
Name: count, dtype: int64

Head:


,usageclass,checkouttype,materialtype,checkoutyear,checkoutmonth,checkouts,title,isbn,creator,subjects,publisher,publicationyear
0,Digital,OverDrive,EBOOK,2020,8,5,I Can Cook Vegan,NaN,Isa Chandra Moskowitz,"Cooking & Food, Nonfiction",ABRAMS,2019
1,Digital,OverDrive,EBOOK,2020,8,5,The Outside,NaN,Ada Hoffmann,"Fiction, Science Fiction","Random House, Inc.",2019
2,Digital,OverDrive,EBOOK,2020,8,16,Blindness,NaN,José Saramago,"Fiction, Literature",Houghton Mifflin Harcourt Trade and Reference,2011
3,Digital,OverDrive,EBOOK,2020,8,5,That Good Night,NaN,Richard Probert,"Fantasy, Fiction",Independent Publishers Group,2016
4,Digital,OverDrive,EBOOK,2020,8,5,Hunting Game,NaN,Helene Tursten,"Fiction, Mystery","Random House, Inc.",2019


In [5]:
# 1. Missingness
print("Nulls per column:")
print(df.isnull().sum())
print("\n% null per column:")
print((df.isnull().sum() / len(df) * 100).round(2))

# 2. Duplicates on the assumed grain
grain_cols = ['title', 'materialtype', 'checkouttype', 'checkoutyear', 'checkoutmonth']
dup_count = df.duplicated(subset=grain_cols).sum()
print(f"\nDuplicates on grain {grain_cols}: {dup_count:,}")

# 3. Checkouts value sanity
print("\nCheckouts summary:")
print(df['checkouts'].describe())

# 4. Cardinality of categoricals
print("\nUnique values in low-card categoricals:")
for col in ['usageclass', 'checkouttype', 'materialtype']:
    print(f"  {col}: {df[col].nunique()} — {df[col].unique().tolist()}")

Nulls per column:
usageclass              0
checkouttype            0
materialtype            0
checkoutyear            0
checkoutmonth           0
checkouts               0
title                   0
isbn               699902
creator             15261
subjects             1788
publisher             977
publicationyear       937
dtype: int64

% null per column:
usageclass          0.00
checkouttype        0.00
materialtype        0.00
checkoutyear        0.00
checkoutmonth       0.00
checkouts           0.00
title               0.00
isbn               52.14
creator             1.14
subjects            0.13
publisher           0.07
publicationyear     0.07
dtype: float64

Duplicates on grain ['title', 'materialtype', 'checkouttype', 'checkoutyear', 'checkoutmonth']: 8,918

Checkouts summary:
count    1.342223e+06
mean     1.225693e+01
std      2.288047e+01
min      5.000000e+00
25%      6.000000e+00
50%      7.000000e+00
75%      1.100000e+01
max      4.903000e+03
Name: checkouts, dtype:

In [ ]:
# Step 3: Monthly checkout volume over time
#
# Each row is one (title x edition x format x month) record, and `checkouts`
# is that month's SUM for the record. Volume is therefore df['checkouts'].sum(),
# never len(df). Rows are not deduplicated: a title appearing twice in a month
# is two editions, and both contribute real checkouts.

import pandas as pd
from pathlib import Path

RAW_PATH = Path('../Data/Raw/seattle_checkouts_2020_2023.csv')
df = pd.read_csv(RAW_PATH, dtype={'isbn': str})

# First-of-month timestamp, e.g. 2020-08 -> 2020-08-01
df['date'] = pd.to_datetime(
    dict(year=df['checkoutyear'], month=df['checkoutmonth'], day=1)
)

# Total checkouts per month
monthly_total = (
    df.groupby('date', as_index=False)['checkouts'].sum()
      .rename(columns={'checkouts': 'total_checkouts'})
)

# Monthly checkouts split into Digital vs Physical
monthly_by_usage = df.pivot_table(
    index='date', columns='usageclass', values='checkouts',
    aggfunc='sum', fill_value=0,
)
monthly_by_usage.columns.name = None

# Yearly checkouts split into Digital vs Physical
yearly_by_usage = df.pivot_table(
    index='checkoutyear', columns='usageclass', values='checkouts',
    aggfunc='sum', fill_value=0,
)
yearly_by_usage.columns.name = None


def head_tail(frame, n=6):
    """First and last n rows of a frame, stacked for a compact preview."""
    return pd.concat([frame.head(n), frame.tail(n)])


print("Monthly total checkouts (first 6 + last 6 months):")
print(head_tail(monthly_total).to_string(index=False))

print("\nMonthly checkouts, Digital vs Physical (first 6 + last 6 months):")
print(head_tail(monthly_by_usage))

print("\nYearly checkouts by usage class:")
print(yearly_by_usage)

# Sanity checks: every aggregate must add back up to the raw total
total_checkouts = df['checkouts'].sum()
assert monthly_total['total_checkouts'].sum() == total_checkouts
assert monthly_by_usage.to_numpy().sum() == total_checkouts
assert yearly_by_usage.to_numpy().sum() == total_checkouts

print("\nSanity checks:")
print(f"  Total checkouts: {total_checkouts:,}")
print(f"  Date range:      {df['date'].min():%Y-%m} to {df['date'].max():%Y-%m}")
print(f"  Unique months:   {df['date'].nunique()} (expected 48)")

In [ ]:
# Step 3b: Investigate the COVID physical checkout gap
#
# Step 3 showed Physical checkouts falling to 0 in April 2020. This cell traces
# the full closure and recovery arc. Values are monthly sums of `checkouts`,
# not row counts.
#
# Caveat: the download kept only records with checkouts >= 5, so a 0 here means
# no physical record reached that floor in the month, not literally zero loans.

import pandas as pd
from pathlib import Path

# Rebuild the monthly pivot only if Step 3 hasn't run in this session
if 'monthly_by_usage' not in globals():
    RAW_PATH = Path('../Data/Raw/seattle_checkouts_2020_2023.csv')
    df = pd.read_csv(RAW_PATH, dtype={'isbn': str})
    df['date'] = pd.to_datetime(
        dict(year=df['checkoutyear'], month=df['checkoutmonth'], day=1)
    )
    monthly_by_usage = df.pivot_table(
        index='date', columns='usageclass', values='checkouts',
        aggfunc='sum', fill_value=0,
    )
    monthly_by_usage.columns.name = None

physical = monthly_by_usage['Physical']

# 1. Every month of 2020
print("Physical checkouts by month, 2020:")
print(physical.loc['2020'].to_frame().to_string())

# 2. When did physical checkouts come back?
zero_months = physical[physical == 0]
if zero_months.empty:
    print("\nPhysical checkouts never hit 0 in 2020-2023.")
else:
    closure_start = zero_months.index.min()
    after_closure = physical.loc[closure_start:]
    reopened = after_closure[after_closure > 0]
    if reopened.empty:
        print(f"\nPhysical checkouts hit 0 in {closure_start:%Y-%m} and never returned.")
    else:
        first_return = reopened.index[0]
        months_at_zero = (after_closure.index < first_return).sum()
        print(f"\nFirst zero month:   {closure_start:%Y-%m}")
        print(f"First return month: {first_return:%Y-%m} "
              f"({reopened.iloc[0]:,} physical checkouts)")
        print(f"Months at zero:     {months_at_zero}")

# 3. Full closure + recovery arc, 2020-2021
arc = physical.loc['2020':'2021']
print(f"\nPhysical checkouts by month, 2020-2021 ({len(arc)} months):")
print(arc.to_frame().to_string())

In [ ]:
# Step 4: Top titles, authors, and format-share evolution
#
# `checkouts` is a per-record monthly sum, so every ranking here sums that
# column rather than counting rows. Editions are collapsed with groupby('title')
# / groupby('creator'); the raw rows are never deduplicated.
#
# Caveats, both visible in the output below:
#   - Title and creator strings follow different conventions per source. Horizon
#     (physical) writes "Lessons in chemistry / Bonnie Garmus." and "Willems, Mo",
#     OverDrive (digital) writes "Lessons in Chemistry: A Novel" and "Mo Willems",
#     and audiobooks append "(unabridged)". So these groupbys collapse editions
#     within a source, not the same work across formats. Normalising that is a
#     cleaning step of its own.
#   - Titles are matched on text alone, so two unrelated works sharing a title
#     would merge. `creator` is 1.14% null, dropped only for the author table.

import pandas as pd
from pathlib import Path

RAW_PATH = Path('../Data/Raw/seattle_checkouts_2020_2023.csv')
df = pd.read_csv(RAW_PATH, dtype={'isbn': str})


def ranked(frame):
    """Number a frame 1..n and label the index 'rank' for display."""
    frame = frame.reset_index(drop=True)
    frame.index = range(1, len(frame) + 1)
    frame.index.name = 'rank'
    return frame


# 1. Top titles overall, all editions and formats combined
title_totals = df.groupby('title')['checkouts'].sum().sort_values(ascending=False)
top_titles = title_totals.head(15)

# The format each of those titles was borrowed in most often
primary_materialtype = (
    df[df['title'].isin(top_titles.index)]
      .groupby(['title', 'materialtype'])['checkouts'].sum()
      .sort_values(ascending=False)
      .reset_index()
      .drop_duplicates('title')          # highest-checkout format per title
      .set_index('title')['materialtype']
)
top_titles = ranked(
    top_titles.rename('total_checkouts').reset_index()
              .assign(primary_materialtype=lambda d: d['title'].map(primary_materialtype))
)

# 2. Top titles within each format
title_by_format = (
    df.groupby(['materialtype', 'title'])['checkouts'].sum()
      .rename('total_checkouts').reset_index()
      .sort_values(['materialtype', 'total_checkouts'], ascending=[True, False])
)
top_by_format = (
    title_by_format.groupby('materialtype').head(5)
                   .assign(rank=lambda d: d.groupby('materialtype').cumcount() + 1)
                   [['materialtype', 'rank', 'title', 'total_checkouts']]
                   .reset_index(drop=True)
)

# 3. Top authors (creator is null on 1.14% of rows, so drop those first)
top_authors = ranked(
    df.dropna(subset=['creator'])
      .groupby('creator')
      .agg(total_checkouts=('checkouts', 'sum'), unique_titles=('title', 'nunique'))
      .sort_values('total_checkouts', ascending=False)
      .head(15)
      .reset_index()
)

# 4. Format share of checkouts per year (% of that year's total)
year_by_format = df.pivot_table(
    index='checkoutyear', columns='materialtype', values='checkouts',
    aggfunc='sum', fill_value=0,
).reindex(columns=['BOOK', 'EBOOK', 'AUDIOBOOK'])
year_by_format.columns.name = None
format_share = (
    year_by_format.div(year_by_format.sum(axis=1), axis=0) * 100
).round(1)

with pd.option_context('display.max_colwidth', 55, 'display.width', 140):
    print("=" * 78)
    print("1. TOP 15 TITLES, 2020-2023 (all formats and editions combined)")
    print("=" * 78)
    print(top_titles.to_string())

    print("\n" + "=" * 78)
    print("2. TOP 5 TITLES PER FORMAT")
    print("=" * 78)
    for materialtype, group in top_by_format.groupby('materialtype'):
        print(f"\n{materialtype}:")
        print(group[['rank', 'title', 'total_checkouts']].to_string(index=False))

    print("\n" + "=" * 78)
    print("3. TOP 15 AUTHORS, 2020-2023")
    print("=" * 78)
    print(top_authors.to_string())

    print("\n" + "=" * 78)
    print("4. FORMAT SHARE OF CHECKOUTS BY YEAR (% of year total)")
    print("=" * 78)
    print(format_share.to_string())

    # Shares are rounded to 1dp, so a row can land on 100.1 instead of 100.0
    print("\nRow totals (sanity check, ~100 each; rounding drift is expected):")
    print(format_share.sum(axis=1).to_string())

In [ ]:
# Step 4b: Title/creator normalization + updated top-N
#
# Step 4 showed the same work splitting across sources, because Horizon
# (physical) and OverDrive (digital) catalogue differently:
#   physical  "Lessons in chemistry / Bonnie Garmus."   creator "Willems, Mo"
#   ebook     "Lessons in Chemistry: A Novel"           creator "Mo Willems"
#   audiobook "Lessons in Chemistry: A Novel (Unabridged)"
#
# clean_title rules, applied in this order:
#   1. lowercase                       — case differs between the two sources
#   2. drop "(unabridged)"/"(abridged)" and any [bracketed] format marker
#      such as "[videorecording]" or "[sound recording]"
#   3. drop a trailing " / author." catalogue suffix (physical records only)
#   4. drop a trailing ": a novel" / ": a memoir", but only when the base
#      title still has 3+ words, so short titles are not gutted
#   5. strip surrounding whitespace and trailing periods
#
# clean_creator rules:
#   "Surname, Forename" and "Surname, Forename, 1966-" become "Forename Surname",
#   birth/death years are dropped, and the result is title-cased. Names without
#   a comma are left as written.
#
# Known limitations, deliberately left in:
#   - Subtitle stripping is a heuristic on two phrases. Other subtitle styles
#     (": a thriller", ": stories") still split, and a work genuinely titled
#     "<something>: A Novel" loses its subtitle. The 3-word guard is why
#     "educated: a memoir" keeps its subtitle below.
#   - .title() mangles internal capitals: "McDonald" becomes "Mcdonald".
#   - Only 4-digit year forms like ", 1966-" or ", 1900-1985" are removed.
#   - Two different works sharing a cleaned title still merge.
# The original `title` and `creator` columns are kept alongside the cleaned ones.

import re
import pandas as pd
from pathlib import Path

RAW_PATH = Path('../Data/Raw/seattle_checkouts_2020_2023.csv')
df = pd.read_csv(RAW_PATH, dtype={'isbn': str})

ABRIDGED_RE = re.compile(r'\s*\((?:un)?abridged\)')
BRACKETED_RE = re.compile(r'\s*\[[^\]]*\]')
CATALOGUE_SUFFIX_RE = re.compile(r'\s+/\s+.*$')
SUBTITLE_RE = re.compile(r'^(?P<base>.+?):\s*a\s+(?:novel|memoir)$')
YEAR_RANGE_RE = re.compile(r'^\d{4}\??-(?:\d{4})?\.?$')


def clean_title_text(title):
    """Normalise one title string (see the rules in the header comment)."""
    if not isinstance(title, str):
        return title
    text = title.lower()
    text = ABRIDGED_RE.sub('', text)
    text = BRACKETED_RE.sub('', text)
    text = CATALOGUE_SUFFIX_RE.sub('', text)
    match = SUBTITLE_RE.match(text)
    if match and len(match.group('base').split()) >= 3:
        text = match.group('base')
    return text.strip().rstrip('.').strip()


def clean_creator_text(creator):
    """Turn "Surname, Forename, 1966-" into "Forename Surname"; keep NaN."""
    if not isinstance(creator, str):
        return creator
    name = creator.strip()
    if ',' in name:
        parts = [p.strip() for p in name.split(',')]
        parts = [p for p in parts if p and not YEAR_RANGE_RE.match(p)]
        if len(parts) >= 2:
            name = f"{parts[1]} {parts[0]}"   # reverse the first two parts
        elif parts:
            name = parts[0]
    return name.strip().title()


def apply_to_unique(series, func):
    """Run func once per distinct value, then map back — far cheaper per row."""
    distinct = series.dropna().unique()
    lookup = pd.Series([func(v) for v in distinct], index=distinct)
    return series.map(lookup)


df['clean_title'] = apply_to_unique(df['title'], clean_title_text)
df['clean_creator'] = apply_to_unique(df['creator'], clean_creator_text)


def ranked(frame):
    """Number a frame 1..n and label the index 'rank' for display."""
    frame = frame.reset_index(drop=True)
    frame.index = range(1, len(frame) + 1)
    frame.index.name = 'rank'
    return frame


# Top titles on the cleaned key
clean_title_totals = (
    df.groupby('clean_title')['checkouts'].sum().sort_values(ascending=False)
)
top_clean_titles = clean_title_totals.head(15)
primary_materialtype = (
    df[df['clean_title'].isin(top_clean_titles.index)]
      .groupby(['clean_title', 'materialtype'])['checkouts'].sum()
      .sort_values(ascending=False)
      .reset_index()
      .drop_duplicates('clean_title')      # highest-checkout format per title
      .set_index('clean_title')['materialtype']
)
top_clean_titles = ranked(
    top_clean_titles.rename('total_checkouts').reset_index()
                    .assign(primary_materialtype=lambda d: d['clean_title'].map(primary_materialtype))
)

# Top creators on the cleaned key
top_clean_creators = ranked(
    df.dropna(subset=['clean_creator'])
      .groupby('clean_creator')
      .agg(total_checkouts=('checkouts', 'sum'),
           unique_titles=('clean_title', 'nunique'))
      .sort_values('total_checkouts', ascending=False)
      .head(15)
      .reset_index()
)

# Spot check: every edition of "Where the Crawdads Sing"
crawdads = df[df['title'].str.contains('Crawdads', case=False, na=False)]
crawdads_mapping = (
    crawdads[['title', 'clean_title', 'materialtype']]
    .drop_duplicates()
    .sort_values(['clean_title', 'materialtype', 'title'])
)
crawdads_totals = (
    crawdads.groupby(['clean_title', 'materialtype'])['checkouts'].sum()
            .rename('total_checkouts').reset_index()
            .sort_values(['clean_title', 'total_checkouts'], ascending=[True, False])
)

with pd.option_context('display.max_colwidth', 60, 'display.width', 150):
    print("=" * 78)
    print("1. NORMALISATION EFFECT (distinct values before vs after)")
    print("=" * 78)
    counts = pd.DataFrame(
        {
            'before': [df['title'].nunique(), df['creator'].nunique()],
            'after': [df['clean_title'].nunique(), df['clean_creator'].nunique()],
        },
        index=['titles', 'creators'],
    )
    counts['merged'] = counts['before'] - counts['after']
    counts['% merged'] = (counts['merged'] / counts['before'] * 100).round(1)
    print(counts.to_string())

    print("\n" + "=" * 78)
    print("2. TOP 15 TITLES ON clean_title")
    print("=" * 78)
    print(top_clean_titles.to_string())

    print("\n" + "=" * 78)
    print("3. TOP 15 CREATORS ON clean_creator")
    print("=" * 78)
    print(top_clean_creators.to_string())

    print("\n" + "=" * 78)
    print("4. SPOT CHECK: 'Crawdads' editions")
    print("=" * 78)
    print(f"\nOriginal -> cleaned ({len(crawdads_mapping)} distinct originals):")
    print(crawdads_mapping.to_string(index=False))
    print("\nCheckouts grouped by (clean_title, materialtype):")
    print(crawdads_totals.to_string(index=False))
    print("\nCombined across formats:")
    print(crawdads.groupby('clean_title')['checkouts'].sum().to_string())

In [ ]:
# Step 5a: Save cleaned dataset
#
# Rebuilds the derived columns from Step 3 (`date`) and Step 4b (`clean_title`,
# `clean_creator`) on a fresh read of the raw file, then writes one cleaned CSV.
# The transformations are repeated here rather than imported so that every
# Step 5 cell runs standalone, in any order, after a kernel restart.
#
# No rows are dropped and nothing is deduplicated: the row count out must equal
# the row count in. The original `title` and `creator` columns are kept next to
# the cleaned versions so any ranking can be traced back to source spellings.

import re
import pandas as pd
from pathlib import Path

RAW_PATH = Path('../Data/Raw/seattle_checkouts_2020_2023.csv')
CLEAN_DIR = Path('../Data/Clean')
CLEAN_PATH = CLEAN_DIR / 'seattle_checkouts_2020_2023_clean.csv'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_PATH, dtype={'isbn': str})
input_rows = len(df)

# --- Step 3 logic: first-of-month timestamp -------------------------------
df['date'] = pd.to_datetime(
    dict(year=df['checkoutyear'], month=df['checkoutmonth'], day=1)
)

# --- Step 4b logic: title and creator normalization -----------------------
ABRIDGED_RE = re.compile(r'\s*\((?:un)?abridged\)')
BRACKETED_RE = re.compile(r'\s*\[[^\]]*\]')
CATALOGUE_SUFFIX_RE = re.compile(r'\s+/\s+.*$')
SUBTITLE_RE = re.compile(r'^(?P<base>.+?):\s*a\s+(?:novel|memoir)$')
YEAR_RANGE_RE = re.compile(r'^\d{4}\??-(?:\d{4})?\.?$')


def clean_title_text(title):
    """Lowercase, drop format markers, catalogue suffix and stock subtitles."""
    if not isinstance(title, str):
        return title
    text = title.lower()
    text = ABRIDGED_RE.sub('', text)
    text = BRACKETED_RE.sub('', text)
    text = CATALOGUE_SUFFIX_RE.sub('', text)
    match = SUBTITLE_RE.match(text)
    if match and len(match.group('base').split()) >= 3:
        text = match.group('base')
    return text.strip().rstrip('.').strip()


def clean_creator_text(creator):
    """Turn "Surname, Forename, 1966-" into "Forename Surname"; keep NaN."""
    if not isinstance(creator, str):
        return creator
    name = creator.strip()
    if ',' in name:
        parts = [p.strip() for p in name.split(',')]
        parts = [p for p in parts if p and not YEAR_RANGE_RE.match(p)]
        if len(parts) >= 2:
            name = f"{parts[1]} {parts[0]}"   # reverse the first two parts
        elif parts:
            name = parts[0]
    return name.strip().title()


def apply_to_unique(series, func):
    """Run func once per distinct value, then map back — far cheaper per row."""
    distinct = series.dropna().unique()
    lookup = pd.Series([func(v) for v in distinct], index=distinct)
    return series.map(lookup)


df['clean_title'] = apply_to_unique(df['title'], clean_title_text)
df['clean_creator'] = apply_to_unique(df['creator'], clean_creator_text)

# --- Write ----------------------------------------------------------------
df.to_csv(CLEAN_PATH, index=False)
output_rows = len(df)

print(f"Input rows:  {input_rows:,}")
print(f"Output rows: {output_rows:,}")
print(f"Row counts match (no rows dropped or deduplicated): {input_rows == output_rows}")

print(f"\nOutput columns ({len(df.columns)}):")
for column in df.columns:
    print(f"  - {column}")

print(f"\nSaved to:   {CLEAN_PATH.resolve()}")
print(f"File exists: {CLEAN_PATH.exists()}")
print(f"File size:   {CLEAN_PATH.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Step 5b: Export aggregation CSVs for Power BI
#
# Reads the cleaned file written by Step 5a rather than any in-memory frame, so
# this cell stands on its own. Five aggregations are written to ../Output/, each
# summing the `checkouts` column — never counting rows.
#
# The two monthly files are padded onto a complete date x category grid. The
# April-July 2020 closure produced no qualifying Physical rows at all, and an
# absent row plots as a gap in the line rather than the zero it really was.
#
# The closing sanity block adds each export back up to the known total and
# checks both grids are full, so a silently dropped row fails loudly here.

import pandas as pd
from pathlib import Path

CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')
OUTPUT_DIR = Path('../Output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_TOTAL = 16_451_540   # established in Step 3
EXPECTED_MONTHS = 48          # 2020-01 through 2023-12

df = pd.read_csv(CLEAN_PATH, dtype={'isbn': str}, parse_dates=['date'])

all_dates = pd.Index(sorted(df['date'].unique()), name='date')


def monthly_grid(category_column):
    """Sum checkouts per (date, category) and pad onto the full grid.

    Every month gets a row for every category; combinations absent from the
    data — a format nobody borrowed that month — come back as 0, not missing.
    """
    aggregated = df.groupby(['date', category_column], as_index=False)['checkouts'].sum()
    categories = pd.Index(sorted(df[category_column].unique()), name=category_column)
    full_grid = pd.MultiIndex.from_product([all_dates, categories])
    padded = aggregated.set_index(['date', category_column]).reindex(full_grid)
    padded['checkouts'] = padded['checkouts'].fillna(0).astype(int)
    return padded.reset_index()


# 1. Monthly checkouts, digital vs physical (48 months x 2 categories = 96 rows)
monthly_by_usageclass = monthly_grid('usageclass')

# 2. Monthly checkouts by material type (48 months x 3 categories = 144 rows)
monthly_by_materialtype = monthly_grid('materialtype')

# 3. Format share of each year's total
format_share_by_year = (
    df.groupby(['checkoutyear', 'materialtype'], as_index=False)['checkouts'].sum()
)
year_totals = format_share_by_year.groupby('checkoutyear')['checkouts'].transform('sum')
format_share_by_year['pct_of_year'] = (
    format_share_by_year['checkouts'] / year_totals * 100
).round(1)

# 4. Top 100 titles on the normalized key, with each title's leading format
title_totals = df.groupby('clean_title')['checkouts'].sum().sort_values(ascending=False)
top_titles = title_totals.head(100)
primary_materialtype = (
    df[df['clean_title'].isin(top_titles.index)]
      .groupby(['clean_title', 'materialtype'])['checkouts'].sum()
      .sort_values(ascending=False)
      .reset_index()
      .drop_duplicates('clean_title')      # highest-checkout format per title
      .set_index('clean_title')['materialtype']
)
top_100_titles = (
    top_titles.rename('total_checkouts').reset_index()
              .assign(primary_materialtype=lambda d: d['clean_title'].map(primary_materialtype))
)
top_100_titles.insert(0, 'rank', range(1, len(top_100_titles) + 1))
top_100_titles = top_100_titles[
    ['rank', 'clean_title', 'primary_materialtype', 'total_checkouts']
]

# 5. Top 50 creators on the normalized key (null creators cannot be ranked)
top_50_creators = (
    df.dropna(subset=['clean_creator'])
      .groupby('clean_creator')
      .agg(total_checkouts=('checkouts', 'sum'),
           unique_titles=('clean_title', 'nunique'))
      .sort_values('total_checkouts', ascending=False)
      .head(50)
      .reset_index()
)
top_50_creators.insert(0, 'rank', range(1, len(top_50_creators) + 1))

exports = {
    'monthly_checkouts_by_usageclass.csv': monthly_by_usageclass,
    'monthly_checkouts_by_materialtype.csv': monthly_by_materialtype,
    'format_share_by_year.csv': format_share_by_year,
    'top_100_titles.csv': top_100_titles,
    'top_50_creators.csv': top_50_creators,
}

print(f"Writing {len(exports)} files to {OUTPUT_DIR.resolve()}\n")
print(f"{'file':<40} {'rows':>7}  {'size':>10}")
print("-" * 60)
for filename, frame in exports.items():
    path = OUTPUT_DIR / filename
    frame.to_csv(path, index=False)
    print(f"{filename:<40} {len(frame):>7,}  {path.stat().st_size / 1e3:>7.1f} KB")

# --- Sanity checks --------------------------------------------------------
usageclass_total = monthly_by_usageclass['checkouts'].sum()
materialtype_total = monthly_by_materialtype['checkouts'].sum()
pct_by_year = format_share_by_year.groupby('checkoutyear')['pct_of_year'].sum()
pct_within_tolerance = pct_by_year.sub(100).abs().le(0.5)

# Padding must produce complete grids, and must not invent checkouts
assert len(monthly_by_usageclass) == EXPECTED_MONTHS * 2, "usageclass grid is not 96 rows"
assert len(monthly_by_materialtype) == EXPECTED_MONTHS * 3, "materialtype grid is not 144 rows"

print("\nSanity checks")
print("-" * 60)
print(f"usageclass total   {usageclass_total:>12,}  vs expected {EXPECTED_TOTAL:,}  "
      f"{'OK' if usageclass_total == EXPECTED_TOTAL else 'MISMATCH'}")
print(f"materialtype total {materialtype_total:>12,}  vs expected {EXPECTED_TOTAL:,}  "
      f"{'OK' if materialtype_total == EXPECTED_TOTAL else 'MISMATCH'}")
print(f"usageclass grid    {len(monthly_by_usageclass):>12,}  rows, expected {EXPECTED_MONTHS * 2}  "
      f"({(monthly_by_usageclass['checkouts'] == 0).sum()} padded to zero)")
print(f"materialtype grid  {len(monthly_by_materialtype):>12,}  rows, expected {EXPECTED_MONTHS * 3}  "
      f"({(monthly_by_materialtype['checkouts'] == 0).sum()} padded to zero)")
print("\npct_of_year per year (100 +/- 0.5 after rounding):")
for year, pct in pct_by_year.items():
    print(f"  {year}  {pct:6.1f}  {'OK' if pct_within_tolerance[year] else 'OUT OF RANGE'}")

In [ ]:
# Step 5c: Draft findings.md
#
# Writes the project write-up to ../findings.md. Nothing is recomputed here:
# every number below was established and sanity-checked in Steps 3 through 4b,
# and is held as literal text so the document cannot drift from what the
# notebook actually printed. Re-running this cell overwrites the file.
#
# Author figures are the post-normalization ones from Step 4b (clean_creator
# and clean_title), so checkout totals and title counts come from the same
# pass. The pre-normalization numbers in Step 4 are not comparable to these.

from pathlib import Path

FINDINGS_PATH = Path('../findings.md')

FINDINGS = '''# Seattle Public Library — Checkouts Analytics (2020–2023)

## Overview

This analysis covers 16,451,540 checkouts recorded by Seattle Public Library across 48 consecutive months, January 2020 through December 2023. The source is the library's "Checkouts by Title" dataset, pulled from the Socrata open-data API and filtered to physical books, ebooks and audiobooks that reached at least five checkouts in a given month, leaving 1,342,223 title-month records. The window brackets the pandemic branch closures and the three years of recovery that followed, so the data captures both the collapse of physical circulation and the format shift that outlasted it: audiobooks overtaking ebooks, digital holding 70% of volume long after the branches reopened, and total circulation growing every year. Built as part of a data analytics portfolio; the notebook, cleaned dataset and Power BI aggregations are listed at the end.

## Key Findings

- **Physical circulation produced no qualifying titles for four consecutive months, April through July 2020**: not one physical title reached five checkouts in a month during the closures, and volume resumed at just 641 in August 2020 before reaching 18,310 that September.
- **Digital still accounted for 70.2% of checkouts in 2023**: three years after the branches reopened, the pandemic shift held — digital came off its 81.4% peak in 2020 but never returned to a print-led split.
- **Audiobooks became the largest format in 2023 at 35.4% of checkouts**: they passed ebooks at 34.8% by 0.6 points, after sitting near 31% for three straight years.
- **Total checkouts grew every year, from 3.28M in 2020 to 4.96M in 2023**: circulation rose 51% across the period, with each year higher than the last despite the closure year at the start.
- **Ebook share fell from 49.7% to 34.8% over four years**: ebooks carried the 2020 closure alone, then gave share back to both audiobooks and returning print.
- **Normalizing titles and creators collapsed 30% of title variants and 25% of creator variants, revealing the true rankings**: the physical and digital catalogs spell the same work differently, so pre-normalization rankings split a single book across three entries and understated its real demand.
- **James Patterson and Taylor Jenkins Reid draw the same volume from catalogs 38 times apart**: Patterson takes 52,775 checkouts across 489 titles against Reid's 52,505 across 13, a 0.5% difference in volume — catalog breadth and per-title velocity arriving at the same place by opposite routes.
- **Where the Crawdads Sing is the most-borrowed title at 27,896 checkouts**: it finishes 2.5% ahead of Braiding Sweetgrass at 27,214, making the top of the list a tie in practice.

## Methodology Notes

- Source: Seattle Public Library "Checkouts by Title", retrieved from the Socrata open-data API at data.seattle.gov.
- Scope: checkout years 2020–2023, restricted to the three main material types — BOOK, EBOOK and AUDIOBOOK.
- Threshold: records below five checkouts in a month were excluded at download, which keeps the extract tractable but means a zero in these tables says no title cleared that floor, not that nothing circulated.
- Grain: one row per title, edition, material type, checkout type, year and month. `checkouts` is a monthly sum, so every volume figure sums that column; row counts are never used as a stand-in. Rows are not deduplicated, since repeated titles are separate editions.
- Normalization: titles lowercased, with "(unabridged)", bracketed format markers, trailing " / Author." catalog suffixes and stock ": a novel" / ": a memoir" subtitles removed; creators converted from "Surname, Forename, 1966-" to "Forename Surname". Title and author rankings quote post-normalization figures throughout.
- Limitation: subtitle stripping is heuristic. It covers two phrases and applies only when three or more words remain, so "Educated: A Memoir" stays separate from a bare "Educated".
- Limitation: creator title-casing flattens internal capitals, rendering "McDonald" as "Mcdonald".

## Files

- `Notebooks/01_eda.ipynb` — full exploratory analysis: API pull, data quality checks, monthly trends, the COVID gap investigation, rankings, and title/creator normalization.
- `Data/Clean/seattle_checkouts_2020_2023_clean.csv` — the filtered dataset with `date`, `clean_title` and `clean_creator` added alongside the original columns.
- `Output/monthly_checkouts_by_usageclass.csv` — monthly checkouts, digital versus physical.
- `Output/monthly_checkouts_by_materialtype.csv` — monthly checkouts by material type.
- `Output/format_share_by_year.csv` — yearly checkouts and percentage share per material type.
- `Output/top_100_titles.csv` — top 100 titles on the normalized key, with each title's primary format.
- `Output/top_50_creators.csv` — top 50 creators on the normalized key, with distinct title counts.
'''

FINDINGS_PATH.write_text(FINDINGS, encoding='utf-8')

print(f"Wrote:  {FINDINGS_PATH.resolve()}")
print(f"Exists: {FINDINGS_PATH.exists()}")
print(f"Size:   {FINDINGS_PATH.stat().st_size / 1e3:.1f} KB")
print(f"Lines:  {len(FINDINGS.splitlines())}")
print(f"Bullets under Key Findings: {FINDINGS.count(chr(10) + '- **')}")